In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import os
import json
import json5
import scipy
from scipy import signal
from math import pi
import math
from typing import Optional
from importlib.resources import files
import hydra
from omegaconf import OmegaConf
import torch.nn.functional as F
from torch import nn
import random
random.seed(114)
import pandas as pd
from glob import glob


attention map

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

# ----------------- 配置与数据处理 -----------------
attn_root = '/data/250010171/code/EigeNet_discriminant/data/attn_map/aa'
layers = [1, 3, 5, 7]
#layers = [0, 2, 4, 6]
#labels = ['S2S', 'S2P', 'P2S', 'P2P']
labels = ['s2S', 'S2s']

# 颜色定义 (学术常用配色)
colors = {
    's2S': '#D62728', # 鲜红色: 代表核心控制位内部交互
    'S2s': '#FF7F0E', # 橙色: 代表从 Source 到 Physical 的注入
    # 'P2S': '#2CA02C', # 绿色: 代表从 Physical 到 Source 的反馈
    # 'P2P': '#1F77B4'  # 深蓝色: 代表物理特征层内部的交互
}

results = {'max': {k: [] for k in labels}, 'mean': {k: [] for k in labels}}

for i in layers:
    attn_path = os.path.join(attn_root, f"layer_{i}.npy")
    # 加载并多头平均
    #attn_map = np.load(attn_path)[0].mean(axis=0)
    attn_map = np.load(attn_path)[0]
    
    parts = {
        's2S': attn_map[:,:2, 27:29],
        'S2s': attn_map[:,27:29, :2],
        # 'P2S': attn_map[1,2:, :2],
        # 'P2P': attn_map[1,2:, 2:]
    }
    
    for k in labels:
        results['max'][k].append(parts[k].max())
        results['mean'][k].append(parts[k].mean())

# ----------------- 绘图逻辑 -----------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 10), sharex=True)
x_ticks = np.arange(len(layers))

def plot_data(ax, data_dict, title, ylabel):
    for label in labels:
        ax.plot(x_ticks, data_dict[label], 
                marker='o', markersize=8, linewidth=2.5,
                label=label, color=colors[label],
                markeredgecolor='white', markeredgewidth=1.5)
    
    ax.set_title(title, fontsize=15, fontweight='bold', pad=15)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(frameon=True, loc='upper right', fontsize=10)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels([f"Layer {i}" for i in layers])

# 执行绘图
plot_data(ax1, results['max'], 'Maximum Attention Intensity', 'Max Score')
plot_data(ax2, results['mean'], 'Average Attention Energy Distribution', 'Mean Score')

ax2.set_xlabel('Transformer Layers (Encoder)', fontsize=13)

plt.tight_layout()
plt.show()